<a href="https://colab.research.google.com/github/pxs1990/NLP_LLM/blob/main/simple_RAG_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Imports
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS


In [ ]:
#Step 2: Initialize AWS Bedrock Models
# Bedrock LLM (Amazon Nova)
llm_nova = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1",
    temperature=0.5,
    max_tokens=200
)
# Bedrock Embeddings (Titan)
embeddings_titan = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v1"
)


In [ ]:
#Step 3: Create Source Documents
texts = [
    "Employees are entitled to 20 days of paid leave per year.",
    "Employees may work from home up to 3 days a week with manager approval.",
    "All employees must complete security training every 6 months."
]
documents = [Document(page_content=text) for text in texts]


In [ ]:
#Step 4: Create Vector Store (FAISS)
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings_titan
)


In [ ]:
#Step 5: Create Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


In [ ]:
#Step 6: Retrieve Relevant Documents
query = "How many days of paid leave do employees get?"
retrieved_docs = retriever.invoke(query)


In [ ]:
#Step 7: Build RAG Prompt (Context + Question)
context = "\n".join(doc.page_content for doc in retrieved_docs)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the provided context."),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])


In [ ]:
#Step 8: Generate Final Answer
response = llm_nova.invoke(
    prompt.format_messages(
        context=context,
        question=query
    )
)
print("RAG Answer:\n", response.content)


In [ ]:
# simple version
# ultra_simple_rag.py
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.chat_models import ChatOllama
from langchain.chains import RetrievalQA

# 1. Load PDF
loader = PyPDFLoader("your_document.pdf")
pages = loader.load()

# 2. Split text
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(pages)

# 3. Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Create vector store
db = FAISS.from_documents(docs, embeddings)

# 5. Create QA chain
llm = ChatOllama(model="llama3.2")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=db.as_retriever(),
    chain_type="stuff"
)

# 6. Ask questions
result = qa_chain.run("Your question here")
print(result)